# When Data Gets Messy

Iris was clean: no missing values, no categories, balanced classes. Real data is never that kind.

**What you'll learn:**
- How the library handles missing values and categorical features automatically
- Why 82% accuracy can be a terrible model
- The evaluate/assess distinction (practice exam vs final exam)
- Deployment gates with `validate()`
- Saving and loading models

**New verbs:** `assess`, `validate`, `save`, `load`

In [1]:
import ml

## 1. A Messy Dataset

Customer churn — 5,000 customers, predicting who will leave. The data has three problems that clean datasets don't.

In [2]:
data = ml.dataset("churn")
data.head()

,age,income,tenure,num_products,account_type,churn
0,56,62369.77,66,4,premium,no
1,69,45850.45,111,1,basic,no
2,46,62572.50,64,1,enterprise,no
3,32,24612.10,26,3,enterprise,no
4,60,36086.57,47,1,basic,no


In [3]:
ml.profile(data, "churn")

── Profile [classification] ────────────
Rows:     5,000
Columns:  6
Target:   churn
Balance:  18% minority class

  ! Imbalanced target: minority class 'yes' is 18% of data
  ! 91 rows (1.8%) have missing values in 'income'
  ! High condition number (54445). Near-collinear features.



Three warnings — three real problems:

1. **Imbalanced target (18%)** — only 18% of customers churned. A model that always predicts "no" would be 82% accurate while catching zero churners.

2. **Missing values in `income`** — 91 rows have NaN. Most sklearn models crash on NaN. mlw handles this automatically: tree-based models pass NaN through natively, others impute with the median.

3. **High condition number** — this warning reflects the scale disparity between features (income ~50,000 vs age ~50), not necessarily that features are correlated. The standard test for multicollinearity is VIF on standardized features, not the condition number of the raw matrix. For tree-based models like XGBoost, feature scale doesn't matter at all.

4. **`account_type` is a string** — three categories (basic, enterprise, premium). The library auto-encodes these into numbers.

## 2. Fit — It Just Works

Same one-line `fit()` as before. The library handles NaN, encodes categories, and picks the right algorithm — no manual preprocessing.

In [4]:
s = ml.split(data, "churn", seed=42)
model = ml.fit(s.train, "churn", seed=42)
print(model)

── Model [xgboost · classification] ────
Target:   churn
Features: 5
Rows:     3,000  (0.02s)
Seed:     42
Hash:     d7d40ce3



No crashes, no errors. The NaN warning from `profile()` told us about the problem; `fit()` handled it silently. All 5 features (including the string column `account_type`) are used.

## 3. Why Accuracy Lies

Let's evaluate.

In [5]:
metrics = ml.evaluate(model, s.valid)
metrics

── Metrics [classification] ────────────
accuracy:   0.7880
f1:         0.1654
precision:  0.2727
recall:     0.1186
roc_auc:    0.6006
(0.02s)

**79% accuracy looks decent.** But look at the other metrics:

- **recall: 0.12** — of all customers who actually churned, the model only caught 12%. It missed 88% of churners.
- **F1: 0.17** — the harmonic mean of precision and recall. Low because recall is abysmal.
- **roc_auc: 0.60** — barely better than a coin flip (0.50).

This is the **accuracy trap**: when one class dominates (82% "no"), a model can score high accuracy by mostly predicting the majority class. It's technically "right" most of the time, but useless for the thing you actually care about — finding the churners.

**Why does ROC-AUC = 0.60 but recall = 0.12?** These measure different things. ROC-AUC uses the model's *probability scores* across all thresholds — it measures whether the model can rank churners higher than non-churners. Recall uses the model's *hard predictions* at the default threshold of 0.5. With 18% churn prevalence, threshold 0.5 is too aggressive — the model needs very high confidence before it predicts "yes", so it almost never does.

On imbalanced data, **F1 and recall matter more than accuracy.** This is why `profile()` warned us upfront.

In [6]:
ml.explain(model)

── Explain [xgboost] ───────────────────
  income        ████████████████████   21.6%
  tenure        ██████████████████▌    20.5%
  age           ██████████████████▌    20.1%
  num_products  █████████████████▌     19.2%
  account_type  █████████████████      18.5%

All features contribute roughly equally — no single dominant predictor. This is common with synthetic/balanced feature sets. In real business data, you'd often see one or two features dominate (like contract length or complaint count).

## 4. Assess — The Final Exam

So far we've used `evaluate()` on the **validation set**. That's the practice exam — call it as many times as you want, tweak your model, try again.

But here's the risk: if you keep tweaking until validation metrics look good, you've indirectly overfit to the validation set. Your reported performance is no longer an honest estimate.

`assess()` solves this. It evaluates on the **test set** — data that was held out from the very beginning. This is the final exam: you take it once, and the score is your honest result.

First, let's retrain on `s.dev` (train + valid combined = 80% of data). Why retrain? We've already chosen our algorithm and tuning approach using the validation set. Now we want the final model to learn from as much data as possible before the test. We keep the same algorithm and hyperparameters — just give it more rows to learn from.

In [7]:
final = ml.fit(s.dev, "churn", seed=42)
print(final)

── Model [xgboost · classification] ────
Target:   churn
Features: 5
Rows:     4,000  (0.01s)
Seed:     42
Hash:     d84278ce



Trained on 4,000 rows instead of 3,000. Now the final exam.

In [8]:
verdict = ml.assess(final, test=s.test)
verdict

── Metrics [classification] ────────────
accuracy:   0.8240
f1:         0.0000
precision:  0.0000
recall:     0.0000
roc_auc:    0.6754
(0.02s)

**F1: 0.0, recall: 0.0** — the model predicts "no" for every single customer. 82% accuracy, zero usefulness.

This is the honest truth about our model on this imbalanced dataset with default settings. The accuracy trap, exposed.

But the ROC-AUC is 0.68 — the model *can* rank churners higher than non-churners. The problem is the default threshold of 0.5. Let's look at what happens if we lower it.

If you call `assess()` again on the same model, you'll get a warning — because looking at test results and going back to tweak defeats the purpose of holding out test data.

In [ ]:
# The model's probability scores still contain signal — let's use them
probs = ml.predict(final, s.test, proba=True)
# Default threshold is 0.5 — try 0.2 instead (closer to the 18% base rate)
fraud_prob = probs["yes"]
custom_preds = (fraud_prob >= 0.2).map({True: "yes", False: "no"})

from sklearn.metrics import recall_score, f1_score, precision_score
y_test = s.test["churn"]
print(f"Threshold 0.5:  recall={recall_score(y_test, final.predict(s.test), pos_label='yes'):.2f}")
print(f"Threshold 0.2:  recall={recall_score(y_test, custom_preds, pos_label='yes'):.2f}, "
      f"precision={precision_score(y_test, custom_preds, pos_label='yes'):.2f}, "
      f"F1={f1_score(y_test, custom_preds, pos_label='yes'):.2f}")

**The fix: lower the threshold.** By moving from 0.5 to 0.2 (closer to the 18% base rate), the model starts actually flagging churners. You trade some precision for recall — catching more churners means some false alarms.

This is the most important lesson for imbalanced classification: **the default threshold of 0.5 is not sacred.** Use `predict(proba=True)` and choose the threshold that matches your business trade-off between catching positives and avoiding false alarms.

## 5. Validate — Deployment Gates

`validate()` sets hard pass/fail rules. No ambiguity — the model either meets your criteria or it doesn't.

In [9]:
gate = ml.validate(final, test=s.test, rules={"accuracy": ">0.75", "roc_auc": ">0.55"})
print(gate)

── Validate [PASSED ✓] ─────────────────
accuracy   0.8240
f1         0.0000
precision  0.0000
recall     0.0000
roc_auc    0.6754



It **passed** — but only because we set weak rules (accuracy > 0.75, roc_auc > 0.55). If we had required F1 > 0.3, it would fail.

This is the power of `validate()`: it forces you to write down *exactly* what "good enough" means before you deploy. No hand-waving, no "looks about right."

**Important:** In practice, set your validation rules *before* seeing test results. If you look at the test metrics first and then set the bar just below them, the gate is circular — you're just rubber-stamping what you already saw. Define "good enough" from business requirements, not from test output.

## 6. Save and Load

Once you have a model worth keeping, save it. mlw uses [skops](https://skops.readthedocs.io/) format — no pickle, no arbitrary code execution.

In [10]:
ml.save(final, "churn_model.ml")

loaded = ml.load("churn_model.ml")
print(f"Loaded:  {loaded.algorithm} · {loaded.features}")

# Verify: same predictions
print(f"Predictions match: {(final.predict(s.test).values == loaded.predict(s.test).values).all()}")

Loaded:  xgboost · ['age', 'income', 'tenure', 'num_products', 'account_type']
Predictions match: True


The `.ml` file contains the trained model, feature encodings, and metadata. Load it anywhere with `ml.load()` — same predictions, guaranteed.

In [11]:
import os
os.remove("churn_model.ml")  # clean up

---

## Recap

Real data taught us three things:

1. **Missing values and categories are handled automatically.** No manual preprocessing needed — `fit()` does it.

2. **Accuracy lies on imbalanced data.** 82% accuracy, 0% recall. Always check F1 and recall when one class dominates. `profile()` warns you about this upfront.

3. **evaluate() and assess() serve different purposes.**
   - `evaluate(model, s.valid)` — practice exam, iterate freely
   - `assess(model, test=s.test)` — final exam, do once
   - `validate(model, test=s.test, rules={...})` — pass/fail gate for deployment

```python
# The production workflow
final   = ml.fit(s.dev, "churn", seed=42)             # retrain on 80%
verdict = ml.assess(final, test=s.test)                 # honest evaluation
gate    = ml.validate(final, test=s.test, rules={...})  # pass/fail
ml.save(final, "model.ml")                              # serialize
loaded  = ml.load("model.ml")                           # deploy
```

**What's next?** In [04_customize.ipynb](04_customize.ipynb), we'll bring our own algorithms, metrics, and splitters — the full customization toolkit.